# Small Trading Company Simulation

This notebook demonstrates loading ERP-like CSV exports, defining a scenario, running the simulator, and reporting KPIs.

## Outline
- Import libraries and set up process model.
- Load sample sales orders CSV via `csv_adapter`.
- Build demand process calibrated to arrivals.
- Run a make-to-stock process for one day.
- Visualize event Gantt chart and compute KPIs.

## Key cells (pseudocode)
```python
from sme_erpsim.io.csv_adapter import load_sales_orders
from sme_erpsim.process.model import Activity, ProcessModel
from sme_erpsim.demand.processes import EmpiricalArrival
from sme_erpsim.simulation.engine import SimulationEngine
from sme_erpsim.simulation.monitors import EventMonitor
from sme_erpsim.kpi.reporting import build_report

orders = load_sales_orders('data/sales_orders.csv')
interarrivals = [
    (orders[i+1].created_at - orders[i].created_at).total_seconds()/3600
    for i in range(len(orders)-1)
]
arrival = EmpiricalArrival(interarrivals)

pm = ProcessModel('trading')
receive = Activity('receive', lambda rng: 0.5)
ship = Activity('ship', lambda rng: 1.0)
pm.add_activity(receive, is_start=True)
pm.add_activity(ship)
pm.add_transition('receive', 'ship')

monitor = EventMonitor()
engine = SimulationEngine(pm, arrival_process=arrival, monitors=[monitor])
engine.run(until=24)
log = monitor.event_log()
report = build_report(log, horizon=24)
print(report.to_markdown())
```
Further cells can plot Gantt charts with `visualization.gantt.plot_gantt(log)`.
